In [1]:
import pandas as pd
import json
import os

races = pd.read_csv('raw_data/races.csv')
circuits = pd.read_csv('raw_data/circuits.csv')

def generate_all_calendars():
    output_dir = 'data/calendars'
    os.makedirs(output_dir, exist_ok=True)
    
    years = races['year'].dropna().unique()
    years.sort()
    
    for year in years:
        year_int = int(year)
        
        # Filtriamo le gare dell'anno e uniamo con i dati dei circuiti
        season_races = races[races['year'] == year_int]
        full_calendar = pd.merge(season_races, circuits, on='circuitId', how='left', suffixes=('_race', '_circuit'))
        
        # Ordiniamo per round in modo sequenziale
        full_calendar = full_calendar.sort_values(by='round')
        
        races_list = []
        for _, row in full_calendar.iterrows():
            # Gestione della data (se presente)
            race_date = row['date'] if pd.notna(row['date']) else "Data non disponibile"
            
            races_list.append({
                "round": int(row['round']),
                "raceName": row['name_race'],
                "circuitName": row['name_circuit'],
                "circuitRef": row['circuitRef'],
                "date": race_date,
                "location": f"{row['location']}, {row['country']}"
            })
            
        output_json = {
            "season": year_int,
            "races": races_list
        }
        
        with open(f'{output_dir}/calendar_{year_int}.json', 'w', encoding='utf-8') as f:
            json.dump(output_json, f, ensure_ascii=False, indent=4)

generate_all_calendars()

In [3]:
pip install Pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
import time
from PIL import Image
from io import BytesIO

def scrape_wikipedia_images_robust(csv_filename, id_col, url_col, output_folder):
    df = pd.read_csv(f'raw_data/{csv_filename}')
    os.makedirs(output_folder, exist_ok=True)
    
    # 1. User-Agent specifico con email fittizia o reale
    headers = {'User-Agent': 'F1DataVizProject/2.0 (Student Project; riccardoparis210401@gmail.com)'}
    
    # 2. Usiamo una Sessione per mantenere viva la connessione TCP
    session = requests.Session()
    session.headers.update(headers)
    
    print(f"Inizio scraping robusto per {csv_filename}...")
    
    for index, row in df.iterrows():
        entity_id = row[id_col]
        wiki_url = row[url_col]
        file_path = os.path.join(output_folder, f"{entity_id}.jpg")
        
        # Salta se esiste già o se manca l'URL
        if os.path.exists(file_path) or pd.isna(wiki_url):
            continue
            
        try:
            response = session.get(wiki_url)
            
            # 3. Gestione esplicita del rate limit sulla pagina
            if response.status_code == 429:
                print(f"[-] Limite superato su {entity_id}. Pausa di 10 secondi...")
                time.sleep(10)
                response = session.get(wiki_url)
                
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            infobox = soup.find('table', class_='infobox')
            
            if infobox:
                img_tag = infobox.find('img')
                if img_tag:
                    img_url = "https:" + img_tag.get('src')
                    
                    if '.svg' in img_url.lower():
                        print(f"[SKIP] Immagine SVG saltata per {entity_id}")
                        continue
                        
                    img_response = session.get(img_url)
                    
                    # Gestione esplicita del rate limit sull'immagine
                    if img_response.status_code == 429:
                        print(f"[-] Limite server immagini superato. Pausa di 10 secondi...")
                        time.sleep(10)
                        img_response = session.get(img_url)
                        
                    img_response.raise_for_status()
                    
                    img = Image.open(BytesIO(img_response.content))
                    
                    if img.mode in ('RGBA', 'P'):
                        background = Image.new('RGB', img.size, (255, 255, 255))
                        if img.mode == 'RGBA':
                            background.paste(img, mask=img.split()[3])
                        else:
                            background.paste(img)
                        img = background
                    elif img.mode != 'RGB':
                        img = img.convert('RGB')
                        
                    img.save(file_path, "JPEG", quality=85)
                    print(f"[OK] Convertito e salvato: {entity_id}.jpg")
                    
                else:
                    print(f"[MISSING] Nessuna immagine per {entity_id}")
            else:
                print(f"[MISSING] Nessuna infobox per {entity_id}")
                
        except Exception as e:
            print(f"[ERROR] Fallimento su {entity_id}: {e}")
            
        # 4. Delay base alzato a 2 secondi
        time.sleep(2)

# Lancia lo scraper (puoi commentare quello dei circuiti finché non finisce quello dei piloti)
#scrape_wikipedia_images_robust('drivers.csv', 'driverRef', 'url', 'assets/img/drivers')
scrape_wikipedia_images_robust('circuits.csv', 'circuitRef', 'url', 'assets/img/circuits')

Inizio scraping robusto per circuits.csv...
[SKIP] Immagine SVG saltata per albert_park
[OK] Convertito e salvato: sepang.jpg
[OK] Convertito e salvato: bahrain.jpg
[SKIP] Immagine SVG saltata per catalunya
[SKIP] Immagine SVG saltata per istanbul
[SKIP] Immagine SVG saltata per monaco
[SKIP] Immagine SVG saltata per villeneuve
[SKIP] Immagine SVG saltata per magny_cours
[OK] Convertito e salvato: silverstone.jpg
[SKIP] Immagine SVG saltata per hockenheimring
[OK] Convertito e salvato: hungaroring.jpg
[SKIP] Immagine SVG saltata per valencia
[SKIP] Immagine SVG saltata per spa
[OK] Convertito e salvato: monza.jpg
[SKIP] Immagine SVG saltata per marina_bay
[SKIP] Immagine SVG saltata per fuji
[SKIP] Immagine SVG saltata per shanghai
[SKIP] Immagine SVG saltata per interlagos
[SKIP] Immagine SVG saltata per indianapolis
[SKIP] Immagine SVG saltata per nurburgring
[OK] Convertito e salvato: imola.jpg
[SKIP] Immagine SVG saltata per suzuka
[SKIP] Immagine SVG saltata per vegas
[OK] Convert

In [7]:
import pandas as pd
import json
import os
import numpy as np

# Funzione di supporto per convertire i tempi stringa in millisecondi
def time_to_ms(t_str):
    try:
        if pd.isna(t_str) or '\\N' in str(t_str): return np.nan
        parts = str(t_str).split(':')
        return (int(parts[0]) * 60 + float(parts[1])) * 1000
    except:
        return np.nan

def generate_race_replays():
    print("Caricamento dataset in corso...")
    lap_times = pd.read_csv('raw_data/lap_times.csv')
    races = pd.read_csv('raw_data/races.csv')
    drivers = pd.read_csv('raw_data/drivers.csv')
    results = pd.read_csv('raw_data/results.csv')
    constructors = pd.read_csv('raw_data/constructors.csv')
    status = pd.read_csv('raw_data/status.csv')
    pit_stops = pd.read_csv('raw_data/pit_stops.csv')
    qualifying = pd.read_csv('raw_data/qualifying.csv')

    output_dir = 'data/replays'
    os.makedirs(output_dir, exist_ok=True)

    races_modern = races[races['year'] >= 1996]

    results_modern = pd.merge(results, races_modern[['raceId', 'year', 'round']], on='raceId', how='inner')
    results_modern = pd.merge(results_modern, drivers[['driverId', 'driverRef']], on='driverId', how='left')
    results_modern = pd.merge(results_modern, constructors[['constructorId', 'constructorRef']], on='constructorId', how='left')
    results_modern = pd.merge(results_modern, status[['statusId', 'status']], on='statusId', how='left')

    df = pd.merge(lap_times, races_modern[['raceId', 'year', 'round']], on='raceId', how='inner')
    df = df.sort_values(by=['raceId', 'lap', 'driverId'])
    df['cumulative_ms'] = df.groupby(['raceId', 'driverId'])['milliseconds'].cumsum()

    grouped_races = df.groupby(['year', 'round'])
    print(f"Generazione JSON e Timeline per {len(grouped_races)} Gran Premi...")

    for (year, rnd), race_data in grouped_races:
        race_id = race_data['raceId'].iloc[0]
        year_int = int(year)
        race_results = results_modern[results_modern['raceId'] == race_id].sort_values(by='positionOrder')
        race_pits = pit_stops[pit_stops['raceId'] == race_id]
        max_lap = race_data['lap'].max()

        # ESTRAZIONE TEMPO QUALIFICA (Per il Meteo)
        race_quali = qualifying[qualifying['raceId'] == race_id]
        best_quali_ms = np.nan
        if not race_quali.empty:
            q_times = []
            for col in ['q1', 'q2', 'q3']:
                if col in race_quali.columns:
                    times = race_quali[col].apply(time_to_ms).dropna()
                    if not times.empty: q_times.append(times.min())
            if q_times: best_quali_ms = min(q_times)
        
        if np.isnan(best_quali_ms):
            best_quali_ms = race_data['milliseconds'].min()

        race_payload = {
            "metadata": { "year": year_int, "round": int(rnd) },
            "timeline": { "sc_periods": [], "vsc_periods": [], "events": [] },
            "laps": []
        }

        # INIZIALIZZAZIONE VARIABILI DI STATO
        previous_top3 = []
        in_sc = False
        in_vsc = False
        sc_start = 0
        vsc_start = 0
        current_weather = "☀️ Sole"
        previous_pack_spread = 0.0  # FIX: Aggiunto qui per evitare il NameError!
        previous_pitting_refs = [] # NUOVO: Memoria dei pit stop del giro precedente

        valid_laps = race_data[race_data['lap'] > 1]
        base_lap_time = valid_laps['milliseconds'].quantile(0.05) if not valid_laps.empty else 100000

        for L in range(0, int(max_lap) + 1):
            standings = []
            pitting_drivers = race_pits[race_pits['lap'] == L]['driverId'].tolist() if not race_pits.empty else []
            
            if L == 0:
                for _, row in race_results.iterrows():
                    grid_pos = int(row['grid']) if pd.notnull(row['grid']) and row['grid'] != 0 else 24
                    standings.append({
                        "driver": str(row['driverRef']), "team": str(row['constructorRef']),
                        "position": grid_pos, "gap_leader": 0.0, "gap_to_ahead": 0.0,
                        "status_type": "grid", "status_text": "In Griglia", "pitting": False
                    })
                standings.sort(key=lambda x: x['position'])
                race_payload["laps"].append({"lap": 0, "weather": current_weather, "standings": standings})
                continue
                
            lap_data = race_data[race_data['lap'] == L]
            if lap_data.empty: continue

            leader_ms = lap_data['cumulative_ms'].min()
            leader_row = lap_data[lap_data['cumulative_ms'] == leader_ms].iloc[0]
            leader_lap_ms = leader_row['milliseconds']

            mins = int(leader_ms // 60000)
            secs = (leader_ms % 60000) / 1000.0
            leader_time_str = f"{mins}:{secs:06.3f}"

            active_drivers = []
            inactive_drivers = []

            for _, res_row in race_results.iterrows():
                d_id = res_row['driverId']
                d_ref = str(res_row['driverRef'])
                driver_lap = lap_data[lap_data['driverId'] == d_id]
                is_pitting = d_id in pitting_drivers
                final_status_str = str(res_row['status'])

                if L == 1 and res_row['laps'] == 0:
                    race_payload["timeline"]["events"].append({
                        "lap": 1, "type": "dns", "driver": d_ref, "text": f"Non Partito ({final_status_str})"
                    })

                if not driver_lap.empty:
                    c_ms = driver_lap.iloc[0]['cumulative_ms']
                    gap_ms = c_ms - leader_ms
                    gap_leader = round(gap_ms / 1000.0, 3)

                    if gap_ms >= leader_lap_ms and leader_lap_ms > 0:
                        status_type = "lapped"
                        status_text = f"+{int(gap_ms // leader_lap_ms)} LAP"
                    else:
                        status_type = "active"
                        status_text = leader_time_str if gap_ms == 0 else f"+{gap_leader}s"

                    active_drivers.append({
                        "driver": d_ref, "team": str(res_row['constructorRef']),
                        "gap_leader": gap_leader, "status_type": status_type,
                        "status_text": status_text, "pos_order": res_row['positionOrder'], "pitting": is_pitting
                    })
                else:
                    is_finished_or_lapped = "Lap" in final_status_str or "Finished" in final_status_str
                    status_type = "lapped" if is_finished_or_lapped else "retired"
                    
                    was_active_last_lap = not race_data[(race_data['lap'] == L-1) & (race_data['driverId'] == d_id)].empty
                    if was_active_last_lap and status_type == "retired" and not ("Lapped" in final_status_str):
                        race_payload["timeline"]["events"].append({
                            "lap": L, "type": "retirement", "driver": d_ref, "text": f"{final_status_str}"
                        })

                    inactive_drivers.append({
                        "driver": d_ref, "team": str(res_row['constructorRef']),
                        "gap_leader": 999.0, "status_type": status_type,
                        "status_text": "+1 LAP" if is_finished_or_lapped else final_status_str,
                        "pos_order": res_row['positionOrder'], "pitting": False
                    })

            active_drivers.sort(key=lambda x: x['gap_leader'])
            inactive_drivers.sort(key=lambda x: x['pos_order'])
            
            combined = active_drivers + inactive_drivers
            
            # --- SALVATAGGIO TOP 3 (Sorpassi vs Pit Stop vs Ritiri) ---
            current_top3 = [d['driver'] for d in active_drivers[:3]]
            
            pitting_refs = [d['driver'] for d in combined if d['pitting']]
            inactive_refs = [d['driver'] for d in inactive_drivers]
            
            # Uniamo i piloti ai box in questo giro con quelli del giro precedente
            recent_pits = pitting_refs + previous_pitting_refs

            if previous_top3 and L > 2:
                for i, driver in enumerate(current_top3):
                    if driver not in previous_top3 or previous_top3.index(driver) > i:
                        
                        if driver in previous_top3:
                            passed_drivers = previous_top3[:previous_top3.index(driver)]
                        else:
                            passed_drivers = previous_top3 
                        
                        # Usiamo la nuova variabile "recent_pits" che copre l'In-Lap e l'Out-Lap
                        is_pit_related = (driver in recent_pits) or any(pd in recent_pits for pd in passed_drivers)
                        is_retirement_related = any(pd in inactive_refs for pd in passed_drivers)
                        
                        if is_retirement_related:
                            race_payload["timeline"]["events"].append({
                                "lap": L, "type": "position_inherited", "driver": driver, "text": f"Sale in P{i+1} (Ritiro avversario 🎁)"
                            })
                        elif is_pit_related:
                            race_payload["timeline"]["events"].append({
                                "lap": L, "type": "overtake_pit", "driver": driver, "text": f"Sale in P{i+1} (Valzer dei Pit Stop 🔄)"
                            })
                        else:
                            race_payload["timeline"]["events"].append({
                                "lap": L, "type": "overtake_track", "driver": driver, "text": f"Sorpasso in pista per la P{i+1}! ⚔️"
                            })
                            
            previous_top3 = current_top3
            
            # Aggiorniamo la memoria per il prossimo giro
            previous_pitting_refs = pitting_refs

            # FIX: Ripristinato il calcolo completo dei gap dinamici
            max_active_gap = active_drivers[-1]['gap_leader'] if active_drivers else 0
            
            gaps_to_ahead_list = []
            for i, d in enumerate(combined):
                d['position'] = i + 1
                if d['gap_leader'] == 999.0:
                    d['gap_leader'] = max_active_gap + (5.0 if d['status_type'] == 'lapped' else 15.0)

                if i > 0 and d['status_type'] == 'active' and combined[i-1]['status_type'] == 'active':
                    d['gap_to_ahead'] = round(d['gap_leader'] - combined[i-1]['gap_leader'], 3)
                    gaps_to_ahead_list.append(d['gap_to_ahead'])
                else:
                    d['gap_to_ahead'] = 0.0
                    
                del d['pos_order']
                standings.append(d)

            # --- EURISTICHE: RED FLAG, SC, VSC, METEO ---
            current_median_ms = lap_data['milliseconds'].median()
            pack_spread = max_active_gap 
            
            delta_spread = pack_spread - previous_pack_spread
            previous_pack_spread = pack_spread

            dynamic_spread_threshold = (base_lap_time / 1000.0) * 0.40

            is_red_flag = current_median_ms > (base_lap_time * 2.5)
            if is_red_flag:
                race_payload["timeline"]["events"].append({
                    "lap": L, "type": "red_flag", "driver": "Direzione Gara", "text": "Bandiera Rossa 🔴"
                })
                if in_sc: 
                    race_payload["timeline"]["sc_periods"].append({"type": "SC", "start": sc_start, "end": L})
                    in_sc = False
                if in_vsc:
                    race_payload["timeline"]["vsc_periods"].append({"type": "VSC", "start": vsc_start, "end": L})
                    in_vsc = False

            vsc_allowed = year_int >= 2015
            is_slow = current_median_ms > (base_lap_time * 1.25)
            
            is_sc_condition = not is_red_flag and is_slow and (pack_spread < dynamic_spread_threshold or delta_spread < -8.0)
            # NUOVA VSC: Lento E sgranato E i distacchi sono "congelati" (oscillano tra -5 e +8 secondi totali)
            is_vsc_condition = not is_red_flag and is_slow and (pack_spread >= dynamic_spread_threshold and -5.0 <= delta_spread <= 8.0) and vsc_allowed

            if is_sc_condition and not in_sc:
                in_sc = True
                sc_start = L
            elif not is_sc_condition and in_sc:
                in_sc = False
                if L - sc_start >= 1: 
                    race_payload["timeline"]["sc_periods"].append({"type": "SC", "start": sc_start, "end": L})

            if is_vsc_condition and not in_vsc and not is_sc_condition:
                in_vsc = True
                vsc_start = L
            elif not is_vsc_condition and in_vsc:
                in_vsc = False
                if L - vsc_start >= 1: 
                    race_payload["timeline"]["vsc_periods"].append({"type": "VSC", "start": vsc_start, "end": L})

            if not (is_sc_condition or is_vsc_condition or is_red_flag) and L > 2:
                fastest_lap_now = lap_data['milliseconds'].min()
                if fastest_lap_now > (best_quali_ms * 1.15):
                    current_weather = "🌧️ Pioggia"
                else:
                    current_weather = "☀️ Sole"

            race_payload["laps"].append({"lap": L, "weather": current_weather, "standings": standings})

        if in_sc: race_payload["timeline"]["sc_periods"].append({"type": "SC", "start": sc_start, "end": int(max_lap)})
        if in_vsc: race_payload["timeline"]["vsc_periods"].append({"type": "VSC", "start": vsc_start, "end": int(max_lap)})

        file_path = os.path.join(output_dir, f"{year}_{rnd}.json")
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(race_payload, f, ensure_ascii=False, separators=(',', ':'))

    print("[OK] Generazione Completata!")

generate_race_replays()

Caricamento dataset in corso...
Generazione JSON e Timeline per 544 Gran Premi...
[OK] Generazione Completata!
